# Lab 2 - Structured Extraction You Can Measure
**Asliddin Tirkashev - RNU, Group 4230Z**

**CPU track, local CPU.** I run this notebook using Ollama's local REST API.
Lab 2 uses all 15 development incidents without a dataset reduction.
I compare ask-and-repair, JSON mode and schema-constrained extraction, then repeat the experiment with the tiny model.

The `results/` files contain actual model responses and measured timings. Gold labels enter evaluation only.
For Colab, change the data setting or upload the supplied ChaosCorp ZIP when requested.


## Step 0 — Setup (run every session, ≈ 5 min)
Run the cells of Step 0 **in order**. You do not need to change anything except the settings cell.
If Colab disconnects, run Step 0 again; your finished results are skipped automatically.

### 0.1 Python packages
Install the dependencies in Colab. In my local run I use the prepared Python environment, so this cell only reports the versions.

In [1]:
import importlib.util, sys, subprocess
try:
    IN_COLAB = importlib.util.find_spec('google.colab') is not None
except ModuleNotFoundError:
    IN_COLAB = False
if IN_COLAB:
    subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', *['requests==2.32.5', 'pydantic==2.14.1', 'pandas==2.2.3']])
import requests, pydantic, pandas as pd
print({'Python': sys.version.split()[0], 'requests': requests.__version__, 'pydantic': pydantic.__version__, 'pandas': pd.__version__})


{'Python': '3.12.14', 'requests': '2.32.5', 'pydantic': '2.14.1', 'pandas': '2.2.3'}


### 0.2 Settings
Both tracks use temperature0 and seed42. GPU uses context4096, maximum512 generated tokens and all GPU layers. CPU uses context2048, maximum200 generated tokens and num_gpu=0. Set GENAI_MODEL to the exact model selected in Lab1.

In [2]:
"""Lab 2: reproducible, resumable Ollama structured extraction measurements."""
from __future__ import annotations

import argparse
import hashlib
import json
import os
import platform
import time
from datetime import datetime, timezone
from pathlib import Path
from typing import Literal, Optional

import pandas as pd
import requests
from pydantic import BaseModel, ConfigDict, Field, ValidationError

HERE = Path.cwd()
DATA = Path(os.environ.get("GENAI_DATA", str(HERE.parent / "data" / "chaoscorp")))
RESULTS = Path(os.environ.get("GENAI_L2_RESULTS", str(HERE / "results")))
TRACK = os.environ.get("GENAI_TRACK", "CPU").upper()
MODEL = os.environ.get("GENAI_MODEL", "qwen3:4b-q4_K_M" if TRACK == "GPU" else "qwen3:1.7b-q4_K_M")
TINY_MODEL = "qwen3:0.6b-q4_K_M"
OLLAMA_URL = os.environ.get("OLLAMA_URL", "http://localhost:11434").rstrip("/")
OPTIONS = {"temperature": 0, "seed": 42, "num_ctx": 4096 if TRACK == "GPU" else 2048,
           "num_predict": 512 if TRACK == "GPU" else 200}
if TRACK == "CPU":
    OPTIONS["num_gpu"] = 0  # A local Mac must not silently use Metal for a declared CPU run.
else:
    OPTIONS["num_gpu"] = 99
IN_COLAB = globals().get("IN_COLAB", False) or bool(os.environ.get("COLAB_RELEASE_TAG") or os.environ.get("COLAB_GPU") or os.environ.get("GENAI_IN_COLAB"))
RUNTIME = os.environ.get("GENAI_RUNTIME", ("Google Colab " if IN_COLAB else "local ") + TRACK)
STRATEGY_NAMES = ("A_retry", "B_json_mode", "C_schema")


USE_DRIVE = False
PRACTICE = "L2"
print("Track:", TRACK, "| model:", MODEL, "| options:", OPTIONS)


Track: CPU | model: qwen3:1.7b-q4_K_M | options: {'temperature': 0, 'seed': 42, 'num_ctx': 2048, 'num_predict': 200, 'num_gpu': 0}


### 0.3 Runtime check
The declared track and actual platform are recorded. GPU track requires a GPU runtime. CPU track handles absence of NVIDIA tooling safely.

In [3]:
import shutil
if shutil.which('nvidia-smi'):
    gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True)
    print(gpu.stdout.strip() or gpu.stderr.strip())
else:
    assert TRACK=='CPU', 'GPU track requires a GPU runtime.'
    print('No NVIDIA GPU detected; local CPU track is selected.')
print(platform.platform())


No NVIDIA GPU detected; local CPU track is selected.
macOS-26.5.1-arm64-arm-64bit


### 0.4 Data and result location
Use the provided ChaosCorp data pack. In Colab this cell asks for the ZIP if it cannot locate the extracted files. Results are saved after every completed incident. Download both this notebook and `results/` before ending a temporary Colab session.

In [4]:
import zipfile
if not DATA.exists():
    candidates = [Path.cwd()/'data/chaoscorp', Path.cwd().parent/'data/chaoscorp', Path.cwd()/'chaoscorp', Path('/content/chaoscorp')]
    DATA = next((p for p in candidates if p.exists()), DATA)
if not DATA.exists() and IN_COLAB:
    from google.colab import files
    uploaded = files.upload()
    zip_path = next(iter(uploaded))
    with zipfile.ZipFile(zip_path) as z:
        assert 'chaoscorp/README.md' in z.namelist(), 'Upload chaoscorp_data_pack_v2.zip.'
        z.extractall('/content')
    DATA = Path('/content/chaoscorp')
assert (DATA/'incidents_dev.json').exists(), f'Data not found: {DATA}. Set GENAI_DATA to the extracted chaoscorp directory.'
RESULTS.mkdir(parents=True, exist_ok=True)
print('Data:', DATA)
print('Results:', RESULTS)


Data: ../data/chaoscorp
Results: results


### 0.5 Ollama connection
Local execution uses the already running Ollama server. The Colab branch installs and starts a local instance. These are local open models and require no paid API key.

In [5]:
if IN_COLAB:
    import os
    if not shutil.which('ollama'):
        subprocess.run(['apt-get','-qq','install','-y','zstd','pciutils'], check=True)
        subprocess.run('curl -fsSL https://ollama.com/install.sh | sh', shell=True, check=True)
    ollama = subprocess.Popen(['ollama','serve'], stdout=open('/content/ollama.log','w'), stderr=subprocess.STDOUT)
    for _ in range(60):
        try:
            response = requests.get(OLLAMA_URL+'/api/version', timeout=2)
            response.raise_for_status()
            break
        except requests.RequestException:
            time.sleep(1)
response = requests.get(OLLAMA_URL+'/api/version', timeout=10)
response.raise_for_status()
print('Ollama:', response.json())


Ollama: {'version': '0.17.4'}


### 0.6 Helper functions (just run this cell)
You will use these helpers in every step. Read the one-line descriptions so you know what they do.

In [6]:
def read_jsonl(path):
    if not Path(path).exists():
        return []
    return [json.loads(line) for line in Path(path).read_text().splitlines() if line.strip()]


def append_jsonl(path, record):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as stream:
        stream.write(json.dumps(record, ensure_ascii=False, allow_nan=False) + "\n")
        stream.flush()
        os.fsync(stream.fileno())


def pull(model):
    response = requests.post(OLLAMA_URL + "/api/pull", json={"model": model, "stream": False}, timeout=3600)
    response.raise_for_status()
    return response.json()


def check_ollama():
    response = requests.get(OLLAMA_URL + "/api/version", timeout=10)
    response.raise_for_status()
    return response.json()


def chat(messages, model, fmt=None):
    body = {"model": model, "messages": messages, "stream": False, "options": OPTIONS,
            "keep_alive": "10m"}
    if model.startswith("qwen3"):
        body["think"] = False
    if fmt is not None:
        body["format"] = fmt
    response = requests.post(OLLAMA_URL + "/api/chat", json=body, timeout=900)
    response.raise_for_status()
    full = response.json()
    return full["message"]["content"], full
print("Helpers ready.")


Helpers ready.


The request helper disables Qwen3 thinking, applies the same options to every strategy, and retains the complete Ollama response for the call log.

### 0.7 Model from Lab 1
I use the exact model chosen in Lab 1. The environment variable `GENAI_MODEL` takes precedence. When a local Lab 1 model card is present, the cell reads it automatically.

In [7]:
if 'GENAI_MODEL' not in os.environ:
    cards = [HERE.parent/'l1/results/model_card.json', HERE.parent/'results/L1/model_card.json']
    card = next((p for p in cards if p.exists()), None)
    if card:
        MODEL = json.loads(card.read_text())['model']
pull(MODEL)
print('Lab 2 main model:', MODEL)


Lab 2 main model: qwen3:1.7b-q4_K_M


## Step 1 — The data and the target record
`incidents_dev.json` contains 15 incident logs. Each has a `raw_log` (the input) and a `gold` block (the correct answers — **only for evaluation, never put them into a prompt**).

In [8]:
INCIDENTS = json.loads((DATA/'incidents_dev.json').read_text())
SEVERITY_GUIDE = (DATA/'policies/policy_severity.md').read_text()
SERVICES = json.loads((DATA/'services.json').read_text())['services']
assert len(INCIDENTS) == 15
print(len(INCIDENTS), 'development incidents')
print(INCIDENTS[3]['raw_log'])
print('Gold is loaded for evaluation, and is not sent to the model.')


15 development incidents
Alert: Database Cluster
ID: 2002
Level: MEDIUM
Latency spiked to 200ms on the Read Replica. 
Re-indexing suggested during maintenance window.
Gold is loaded for evaluation, and is not sent to the model.


Some incidents are deliberately hard. Their `gold.hard_case_tags` say why:

| Tag | Meaning | Example |
|---|---|---|
| `H1_missing_field` | the ID or the severity label is missing | no ID in the log → correct `incident_id` is `null` |
| `H2_contradictory_severity` | the written label contradicts the impact | label "Low" but data corruption → the guide's override makes it **High** |
| `H3_format_noise` | typos, odd formatting, broken lines | "glich", lowercase dumps |
| `H4_ambiguous_service` | the service is named by an alias | "Database Cluster" → `DatabaseCluster` |
| `H5_multi_service` | two services are mentioned | pick the one where the fault is |

In [9]:
import collections
print("Hard-case tags in the 15 incidents:", dict(collections.Counter(t for i in INCIDENTS for t in i["gold"]["hard_case_tags"])))

Hard-case tags in the 15 incidents: {'H4_ambiguous_service': 2, 'H3_format_noise': 4, 'H5_multi_service': 1, 'H1_missing_field': 1, 'H2_contradictory_severity': 1}


The **target record** is defined with Pydantic. `Literal[...]` means only these exact values are allowed.

In [10]:
ServiceName = Literal["PaymentGateway_v2", "EmailNotifier", "AuthServer", "DatabaseCluster", "Frontend_React",
                      "UserProfileDB", "SearchEngine", "payment-processor", "InventoryAPI", "Recommendations-AI", "UNKNOWN"]


class Extraction(BaseModel):
    model_config = ConfigDict(extra="forbid", strict=True)
    incident_id: Optional[int] = Field(description="ID in the log, or null if there is none")
    severity: Literal["High", "Medium", "Low"]
    service_name: ServiceName
    summary: str = Field(description="at most 12 words")


def load_data():
    incidents = json.loads((DATA / "incidents_dev.json").read_text())
    guide = (DATA / "policies" / "policy_severity.md").read_text()
    services = json.loads((DATA / "services.json").read_text())["services"]
    service_lines = "\n".join(f"- {s['service_name']} (also called: {', '.join(s['aliases'])})" for s in services)
    system_prompt = f"""You extract incident records for Chaos Corp.
Apply this severity guide exactly (including the upward overrides):
{guide}

Valid service names (map aliases to the canonical name; use UNKNOWN if no service matches):
{service_lines}

Return JSON with exactly these fields: incident_id (integer or null), severity (High|Medium|Low), service_name, summary (at most 12 words).
Return only JSON without markdown or explanations."""
    assert len(incidents) == 15
    return incidents, system_prompt


def validate(text):
    try:
        return Extraction.model_validate_json(text), None
    except ValidationError as error:
        return None, str(error)


def words_ok(record):
    if isinstance(record, Extraction):
        record = record.model_dump()
    return record is not None and len(record["summary"].split()) <= 12
INCIDENTS, SYSTEM_PROMPT = load_data()
print(SYSTEM_PROMPT[:600])


You extract incident records for Chaos Corp.
Apply this severity guide exactly (including the upward overrides):
# Chaos Corp — Severity Guide
doc_id: policy_severity

## s1_label_mapping — Explicit labels
Map the label written in the log to exactly one of High | Medium | Low:
- **High:** critical, critical failure, sev1, sev-1, p1, high, urgent
- **Medium:** medium, sev2, sev-2, p2, moderate
- **Low:** low, low priority, sev3, sev4, p3, p4, minor
Labels are case-insensitive and may be abbreviated (e.g., `Sev: HIGH`, `Level: MEDIUM`, `| Low Priority`).

## s2_overrides — Upward overrides
Raise


## Step 2 — Strategy A: Ask & Repair
We ask for JSON in the prompt. If the answer is not valid, we send the **error message back** and ask the model to fix it — at most 3 retries.

### ✏️ YOUR TURN 1 — write the repair message
Complete `repair_message(error)`. It returns the text we send back to the model. It must:
1. say that its **JSON was invalid**,
2. include the **error text** (the variable `error`),
3. ask for **only** the corrected JSON (no explanations, no markdown).

Hint: use an f-string, e.g. `f"... {error} ..."`.

In [11]:
def repair_message(error):
    return f"Your JSON was invalid: {error}. Return only corrected JSON, with no explanations or markdown."

In [12]:
# ✅ Check (do not edit)
msg = repair_message("severity: Input should be 'High', 'Medium' or 'Low'")
assert msg, "repair_message returned an empty string."
assert "severity: Input should be" in msg, "The message must contain the error text: put {error} inside your f-string."
assert "json" in msg.lower(), "The message must mention JSON."
print("✅ Repair message OK:", msg)

✅ Repair message OK: Your JSON was invalid: severity: Input should be 'High', 'Medium' or 'Low'. Return only corrected JSON, with no explanations or markdown.


Now the full strategy (just run it). `run_strategy` works like a short conversation:
1. send the system prompt + the log → the model answers;
2. `validate` the answer; if it is valid, stop;
3. otherwise add the model's answer **and** your `repair_message` to the conversation and ask again — at most 3 more times.

It records every attempt, so we can later count how many calls were needed and how long they took.

In [13]:
def config_id(model, system_prompt):
    payload = {"model": model, "options": OPTIONS, "prompt": system_prompt,
               "schema": Extraction.model_json_schema(), "version": 1}
    return hashlib.sha256(json.dumps(payload, sort_keys=True).encode()).hexdigest()


def dataset_identity(incidents):
    """Cache only model inputs; gold labels remain evaluation-only."""
    inputs=[{'incident_key':row['incident_key'],'raw_log':row['raw_log']} for row in incidents]
    return hashlib.sha256(json.dumps(inputs,sort_keys=True,ensure_ascii=False).encode()).hexdigest()


def ensure_dataset_identity(folder,incidents):
    path=Path(folder)/'dataset_manifest.json'
    expected={'version':1,'inputs_sha256':dataset_identity(incidents),
              'incident_keys':[row['incident_key'] for row in incidents]}
    if path.exists():
        if json.loads(path.read_text())['identity']!=expected:
            raise ValueError('Cached dataset identity differs; use a new results folder.')
    else:
        if any(read_jsonl(Path(folder)/f'{name}.jsonl') for name in STRATEGY_NAMES):
            raise ValueError('Legacy cached dataset has no identity manifest; audit it or use a new results folder.')
        path.write_text(json.dumps({'identity':expected,'established_utc':datetime.now(timezone.utc).isoformat(),
                                   'established_by':'before model calls'},indent=2))


def run_strategy(incident, model, strategy, system_prompt, call_log_path=None):
    if strategy not in STRATEGY_NAMES:
        raise ValueError(strategy)
    fmt = None if strategy == "A_retry" else ("json" if strategy == "B_json_mode" else Extraction.model_json_schema())
    max_retries = 1 if strategy == "C_schema" else 3
    # Only raw_log enters the model. Gold is reserved for evaluation below.
    messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": incident["raw_log"]}]
    attempts, final = [], None
    experiment_id = config_id(model, system_prompt)
    for attempt in range(1 + max_retries):
        started = time.perf_counter()
        full, raw, error, rec = None, None, None, None
        try:
            raw, full = chat(messages, model=model, fmt=fmt)
            rec, error = validate(raw)
        except (requests.RequestException, KeyError, ValueError) as failure:
            error = f"{type(failure).__name__}: {failure}"
        latency = time.perf_counter() - started
        entry = {"experiment_id": experiment_id, "model": model, "strategy": strategy,
                 "incident_key": incident["incident_key"], "attempt": attempt + 1,
                 "timestamp_utc": datetime.now(timezone.utc).isoformat(), "latency_s": latency,
                 "raw": raw, "error": error, "schema_valid": rec is not None,
                 "summary_words_ok": words_ok(rec), "response": full}
        attempts.append(entry)
        if call_log_path is not None:
            append_jsonl(call_log_path, entry)
        if rec is not None:
            final = rec.model_dump()
            break
        if raw is not None:
            messages.append({"role": "assistant", "content": raw})
        messages.append({"role": "user", "content": repair_message(error)})
    return {"experiment_id": experiment_id, "model": model, "strategy": strategy,
            "incident_key": incident["incident_key"], "attempts": attempts, "calls": len(attempts),
            "latency_s": sum(a["latency_s"] for a in attempts), "final": final}


def run_all(folder, model, incidents, system_prompt):
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    ensure_dataset_identity(folder,incidents)
    experiment_id = config_id(model, system_prompt)
    for strategy in STRATEGY_NAMES:
        path = folder / f"{strategy}.jsonl"
        existing = read_jsonl(path)
        if any(row.get("experiment_id") != experiment_id for row in existing):
            raise ValueError(f"Cached configuration differs: {path}; use a new results folder.")
        done = {row["incident_key"] for row in existing}
        for inc in incidents:
            if inc["incident_key"] not in done:
                result = run_strategy(inc, model, strategy, system_prompt, folder / "calls.jsonl")
                append_jsonl(path, result)
                print(f"{model} {strategy} {inc['incident_key']}: calls={result['calls']}, valid={result['final'] is not None}", flush=True)
        print(strategy, "completed", len(read_jsonl(path)), "/", len(incidents), flush=True)
def strategy_a(inc):
    return run_strategy(inc, MODEL, 'A_retry', SYSTEM_PROMPT)
print('A: ordinary JSON prompt, Pydantic validation, at most three repairs.')


A: ordinary JSON prompt, Pydantic validation, at most three repairs.


## Step 3 — Strategy B: JSON mode
Ollama can force the output to be **syntactically valid JSON** (`format="json"`). It does *not* know our schema, so we still validate and repair.

### ✏️ YOUR TURN 2 — switch on JSON mode
Complete `strategy_b`. Look at `strategy_a` in Step 2: it calls `run_strategy(inc, fmt=None)`. Strategy B is the same call with JSON mode switched on, i.e. `fmt="json"`. One line.

In [14]:
def strategy_b(inc):
    return run_strategy(inc, MODEL, 'B_json_mode', SYSTEM_PROMPT)


In [15]:
assert callable(strategy_b)
print('B: format="json", the same Pydantic validation and repair loop.')


B: format="json", the same Pydantic validation and repair loop.


## Step 4 - Strategy C: Schema-constrained decoding
I pass `Extraction.model_json_schema()` as the Ollama REST `format`. This is the alternative explicitly allowed by the brief. I validate the returned record with the same Pydantic model and measure the word limit separately.

In [16]:
def strategy_c(inc):
    return run_strategy(inc, MODEL, 'C_schema', SYSTEM_PROMPT)
print('C: JSON-schema-constrained decoding, followed by validation.')


C: JSON-schema-constrained decoding, followed by validation.


### Run all three strategies on all 15 incidents (≈ 2–5 min on a T4)
Results are saved after every incident, so after a disconnect this cell continues where it stopped.

In [17]:
MAIN = RESULTS/'main'
STRATEGIES = {'A_retry': strategy_a, 'B_json_mode': strategy_b, 'C_schema': strategy_c}
run_all(MAIN, MODEL, INCIDENTS, SYSTEM_PROMPT)


A_retry completed 15 / 15


B_json_mode completed 15 / 15


C_schema completed 15 / 15


## Step 5 — Evaluate
### The metrics table (just run it)
| Row | Meaning |
|---|---|
| valid first try % | the first answer was already a valid record |
| valid after retries % | a valid record was obtained in the end |
| mean calls | LLM calls per incident (1 = no repair needed) |
| mean latency s | seconds per incident, all calls together |
| `<field>` acc % | the final value equals the gold answer (an invalid output counts as wrong) |
| summary <=12 words % | the summary respects the word limit |

In [18]:
FIELDS = ("incident_id", "severity", "service_name")


def evaluate(strategy, folder, incidents):
    rows = read_jsonl(Path(folder) / f"{strategy}.jsonl")
    gold = {incident["incident_key"]: incident["gold"] for incident in incidents}
    n = len(incidents)
    if len(rows) != n or len({row["incident_key"] for row in rows}) != n:
        raise ValueError(f"Expected {n} unique results for {strategy}; found {len(rows)}")
    accuracy = {f: 100 * sum(row["final"] is not None and row["final"][f] == gold[row["incident_key"]][f]
                             for row in rows) / n for f in FIELDS}
    return {"strategy": strategy,
            "valid first try %": 100 * sum(row["attempts"][0]["schema_valid"] for row in rows) / n,
            "valid after retries %": 100 * sum(row["final"] is not None for row in rows) / n,
            "mean calls": sum(row["calls"] for row in rows) / n,
            "mean latency s": sum(row["latency_s"] for row in rows) / n,
            **{f"{f} acc %": value for f, value in accuracy.items()},
            "summary <=12 words %": 100 * sum(words_ok(row["final"]) for row in rows) / n}


def field_explanation(field, incident, predicted, valid):
    if not valid:
        return "No schema-valid final record; every target field counts as wrong."
    tags = incident["gold"]["hard_case_tags"]
    if field == "incident_id":
        if predicted is None and incident['gold']['incident_id'] is not None:
            return f"The output omits the explicit incident identifier {incident['gold']['incident_id']} present in the log."
        return "The record invents or misreads an incident identifier; missing IDs must remain null."
    if field == "severity":
        if "H2_contradictory_severity" in tags:
            return "The written label conflicts with impact; the severity guide's upward override takes priority."
        return "The severity does not follow the label mapping or no-label impact rules."
    if "H5_multi_service" in tags:
        return "Several services are mentioned; the output selects a different service from the one at fault."
    if "H4_ambiguous_service" in tags:
        return "The alias is not mapped to the correct canonical service."
    return "The selected canonical service does not match the service affected in the log."


def error_table(folder, incidents):
    errors = []
    by_key = {incident["incident_key"]: incident for incident in incidents}
    for strategy in STRATEGY_NAMES:
        for row in read_jsonl(Path(folder) / f"{strategy}.jsonl"):
            inc = by_key[row["incident_key"]]
            final, g = row["final"], inc["gold"]
            for field in FIELDS:
                predicted = final[field] if final is not None else None
                if final is None or predicted != g[field]:
                    errors.append({"strategy": strategy, "incident_key": inc["incident_key"],
                                   "field": field, "predicted": predicted, "gold": g[field],
                                   "valid_output": final is not None,
                                   "hard_case_tags": ", ".join(g["hard_case_tags"]),
                                   "explanation": field_explanation(field, inc, predicted, final is not None)})
    return pd.DataFrame(errors, columns=["strategy", "incident_key", "field", "predicted", "gold", "valid_output", "hard_case_tags", "explanation"])


def metric_table(folder, incidents):
    return pd.DataFrame([evaluate(name, folder, incidents) for name in STRATEGY_NAMES]).set_index("strategy").T


def summary_limit_violations(folder):
    rows = []
    for strategy in STRATEGY_NAMES:
        for row in read_jsonl(Path(folder) / f'{strategy}.jsonl'):
            final = row['final']
            if final is not None and not words_ok(final):
                rows.append({'strategy': strategy, 'incident_key': row['incident_key'],
                             'summary': final['summary'], 'word_count': len(final['summary'].split())})
    return pd.DataFrame(rows, columns=['strategy', 'incident_key', 'summary', 'word_count'])


def save_tables(folder, incidents):
    folder = Path(folder)
    metric_table(folder, incidents).to_csv(folder / "metrics.csv")
    error_table(folder, incidents).to_csv(folder / "errors.csv", index=False)
    summary_limit_violations(folder).to_csv(folder / 'summary_limit_violations.csv', index=False)
    calls = read_jsonl(folder / "calls.jsonl")
    pd.DataFrame([{k: v for k, v in call.items() if k != "response"} for call in calls]).to_csv(folder / "calls.csv", index=False)
metrics = metric_table(MAIN, INCIDENTS)
display(metrics.round(2))
pd.set_option('display.max_rows', 200)
pd.set_option('display.max_colwidth', 120)
call_log = pd.DataFrame([{k:v for k,v in call.items() if k!='response'} for call in read_jsonl(MAIN/'calls.jsonl')])
display(call_log[['model','strategy','incident_key','attempt','latency_s','schema_valid','error','raw']])
save_tables(MAIN, INCIDENTS)
print('Full call logs:', MAIN/'calls.jsonl')


strategy,A_retry,B_json_mode,C_schema
valid first try %,100.00,100.00,100.00
valid after retries %,100.00,100.00,100.00
mean calls,1.00,1.00,1.00
mean latency s,1.03,1.05,1.12
incident_id acc %,93.33,93.33,93.33
severity acc %,100.00,100.00,100.00
service_name acc %,100.00,100.00,100.00
summary <=12 words %,100.00,100.00,100.00


,model,strategy,incident_key,attempt,latency_s,schema_valid,error,raw
0,qwen3:1.7b-q4_K_M,A_retry,ORIG-1044,1,2.362199,True,None,"{\n ""incident_id"": 1044,\n ""severity"": ""High"",\n ""service_name"": ""PaymentGateway_v2"",\n ""summary"": ""API endpoint..."
1,qwen3:1.7b-q4_K_M,A_retry,ORIG-9901,1,0.640608,True,None,"{\n ""incident_id"": null,\n ""severity"": ""Low"",\n ""service_name"": ""EmailNotifier"",\n ""summary"": ""Delayed reset pas..."
2,qwen3:1.7b-q4_K_M,A_retry,ORIG-4055,1,0.781649,True,None,"{\n ""incident_id"": 4055,\n ""severity"": ""High"",\n ""service_name"": ""AuthServer"",\n ""summary"": ""Memory leak in Auth..."
3,qwen3:1.7b-q4_K_M,A_retry,ORIG-2002,1,0.856913,True,None,"{\n ""incident_id"": 2002,\n ""severity"": ""Medium"",\n ""service_name"": ""DatabaseCluster"",\n ""summary"": ""Latency spik..."
4,qwen3:1.7b-q4_K_M,A_retry,ORIG-5500,1,0.934429,True,None,"{\n ""incident_id"": 5500,\n ""severity"": ""Low"",\n ""service_name"": ""Frontend_React"",\n ""summary"": ""CSS not loading ..."
5,qwen3:1.7b-q4_K_M,A_retry,ORIG-8812,1,0.937706,True,None,"{\n ""incident_id"": 8812,\n ""severity"": ""High"",\n ""service_name"": ""UserProfileDB"",\n ""summary"": ""Data corruption ..."
6,qwen3:1.7b-q4_K_M,A_retry,ORIG-3321,1,0.984964,True,None,"{\n ""incident_id"": 3321,\n ""severity"": ""Low"",\n ""service_name"": ""SearchEngine"",\n ""summary"": ""SearchEngine down ..."
7,qwen3:1.7b-q4_K_M,A_retry,ORIG-1111,1,0.817808,True,None,"{\n ""incident_id"": 1111,\n ""severity"": ""High"",\n ""service_name"": ""payment-processor"",\n ""summary"": ""Duplicate tr..."
8,qwen3:1.7b-q4_K_M,A_retry,ORIG-7734,1,0.911410,True,None,"{\n ""incident_id"": 7734,\n ""severity"": ""Medium"",\n ""service_name"": ""InventoryAPI"",\n ""summary"": ""Stock counts mi..."
9,qwen3:1.7b-q4_K_M,A_retry,ORIG-6005,1,0.934857,True,None,"{\n ""incident_id"": 6005,\n ""severity"": ""Low"",\n ""service_name"": ""Recommendations-AI"",\n ""summary"": ""Model infere..."


Full call logs: results/main/calls.jsonl


### The error table
One row per **wrong field**: which strategy, which incident, what the model said (`predicted`) and the correct answer (`gold`).
`valid_output = True` means the output was perfectly valid JSON **and still wrong**: these are the *valid-but-wrong* cases.

In [19]:
err_df = error_table(MAIN, INCIDENTS)
display(err_df if len(err_df) else 'No incorrect target fields in this run.')
print('Valid-but-wrong field rows:', int(err_df['valid_output'].sum()) if len(err_df) else 0)
summary_errors = summary_limit_violations(MAIN)
display(summary_errors if len(summary_errors) else 'All main-model summaries contain at most 12 words.')


,strategy,incident_key,field,predicted,gold,valid_output,hard_case_tags,explanation
0,A_retry,ORIG-9901,incident_id,None,9901,True,,The output omits the explicit incident identifier 9901 present in the log.
1,B_json_mode,ORIG-9901,incident_id,None,9901,True,,The output omits the explicit incident identifier 9901 present in the log.
2,C_schema,ORIG-9901,incident_id,None,9901,True,,The output omits the explicit incident identifier 9901 present in the log.


Valid-but-wrong field rows: 3


'All main-model summaries contain at most 12 words.'

## Step 5B — Stress test with a tiny model (≈ 1–2 min)
Your model probably produced valid JSON almost every time with all three strategies. Does that mean the strategy does not matter?
We repeat the experiment with a **very small model** (`qwen3:0.6b`, 0.5 GB), which makes more mistakes. Its results are saved in a **separate folder**, so your main results stay unchanged. Just run the cells.

In [20]:
STRESS = RESULTS/'stress_tiny'
pull(TINY_MODEL)
run_all(STRESS, TINY_MODEL, INCIDENTS, SYSTEM_PROMPT)
save_tables(STRESS, INCIDENTS)
print('Main model retained:', MODEL)


A_retry completed 15 / 15


B_json_mode completed 15 / 15


C_schema completed 15 / 15


Main model retained: qwen3:1.7b-q4_K_M


### Your model vs the tiny model — side by side

In [21]:
compare = pd.concat({f'main ({MODEL})': metric_table(MAIN, INCIDENTS), f'tiny ({TINY_MODEL})': metric_table(STRESS, INCIDENTS)}, axis=1)
display(compare.round(2))
tiny_err = error_table(STRESS, INCIDENTS)
display(tiny_err if len(tiny_err) else 'No incorrect target fields in this stress run.')
print('Tiny valid-but-wrong field rows:', int(tiny_err['valid_output'].sum()) if len(tiny_err) else 0)
tiny_summary_errors = summary_limit_violations(STRESS)
display(tiny_summary_errors if len(tiny_summary_errors) else 'All tiny-model summaries contain at most 12 words.')


main (qwen3:1.7b-q4_K_M)                       \
strategy                               A_retry B_json_mode C_schema   
valid first try %                       100.00      100.00   100.00   
valid after retries %                   100.00      100.00   100.00   
mean calls                                1.00        1.00     1.00   
mean latency s                            1.03        1.05     1.12   
incident_id acc %                        93.33       93.33    93.33   
severity acc %                          100.00      100.00   100.00   
service_name acc %                      100.00      100.00   100.00   
summary <=12 words %                    100.00      100.00   100.00   

                      tiny (qwen3:0.6b-q4_K_M)                       
strategy                               A_retry B_json_mode C_schema  
valid first try %                       100.00      100.00   100.00  
valid after retries %                   100.00      100.00   100.00  
mean calls                                1.00        1.00     1.00  
mean latency s                            0.81        0.80     0.85  
incident_id acc %                        93.33       93.33    93.33  
severity acc %                           93.33       93.33    93.33  
service_name acc %                      100.00      100.00   100.00  
summary <=12 words %                     93.33       93.33    93.33

,strategy,incident_key,field,predicted,gold,valid_output,hard_case_tags,explanation
0,A_retry,NEW-1-12,incident_id,123456,None,True,H1_missing_field,The record invents or misreads an incident identifier; missing IDs must remain null.
1,A_retry,NEW-2-07,severity,Medium,High,True,"H2_contradictory_severity, H3_format_noise, H4_ambiguous_service",The written label conflicts with impact; the severity guide's upward override takes priority.
2,B_json_mode,NEW-1-12,incident_id,123456,None,True,H1_missing_field,The record invents or misreads an incident identifier; missing IDs must remain null.
3,B_json_mode,NEW-2-07,severity,Medium,High,True,"H2_contradictory_severity, H3_format_noise, H4_ambiguous_service",The written label conflicts with impact; the severity guide's upward override takes priority.
4,C_schema,NEW-1-12,incident_id,123456,None,True,H1_missing_field,The record invents or misreads an incident identifier; missing IDs must remain null.
5,C_schema,NEW-2-07,severity,Medium,High,True,"H2_contradictory_severity, H3_format_noise, H4_ambiguous_service",The written label conflicts with impact; the severity guide's upward override takes priority.


Tiny valid-but-wrong field rows: 6


,strategy,incident_key,summary,word_count
0,A_retry,NEW-1-08,"AuthServer OOMKilled twice within 40m on auth pods 2 and 7, with no credential leak detected.",16
1,B_json_mode,NEW-1-08,"AuthServer OOMKilled twice within 40m on auth pods 2 and 7, with no credential leak detected.",16
2,C_schema,NEW-1-08,"AuthServer OOMKilled twice within 40m on auth pods 2 and 7, with no credential leak detected.",16


**What to look for:**
- **validity:** does strategy A now need repairs (`mean calls` > 1) or fail completely, while B and C stay valid?
- **latency:** what does "ask and repair" cost compared with enforced formats?
- **correctness:** do B and C make *fewer content errors* than A, or only fewer *format* errors? Look for invented values (for example an `incident_id` that is not in the log).

## Step 6 — Analysis
### ✏️ YOUR TURN 3 — answer in your own words (≈ 250 words in total)
1. Did B and C reduce the number of calls compared with A? What happened to latency?
2. Which strategy had the best **accuracy** (not validity)? Are the errors concentrated on some hard-case tags (H1 missing field, H2 contradictory severity, H3 format noise, H4 alias, H5 multi-service)?
3. Pick **one valid-but-wrong** case from the error table. Why is it wrong, and what could catch such errors in a real system?
4. Which strategy would you use in a backend batch job, and which in a user-facing chatbot? Why?
5. **Step 5B:** compare your model with the tiny model. What did enforcing the format (B, C) fix, and what did it *not* fix? Give one concrete example from the tiny model's error table.

**1.** I measured one call per incident for every strategy. All 15 records were schema-valid on the first attempt, so neither JSON mode nor constrained decoding reduced retries in this sample. Mean total latency was 1.03 seconds for A, 1.05 for B and 1.12 for C. The schema guarantee had a small measured cost.

**2.** All three strategies tied: severity and service accuracy were 15/15, while incident ID accuracy was 14/15. Every strategy made the same omission in ORIG-9901. This incident has no hard-case tag, so the main-model errors were not concentrated in H1-H5 cases. The result also shows why format validity and field accuracy need separate measurements.

**3.** In ORIG-9901, the source explicitly says Ticket #9901, but every strategy returned incident_id=null. The JSON still satisfies the optional-integer schema. I would check the extracted identifier against a source span and route contradictions for review. Repeating schema validation alone would not catch this omission.

**4.** I would choose C for a backend batch job because its output contract is explicit. For an interactive chatbot, I would also start with C and provide a clear fallback after failure. In this run its mean latency remained close to A and B, but I would measure larger workloads before setting a response-time target.

**5.** The tiny model also achieved 100% first-try schema validity. Its ID and severity accuracy were each 14/15, and only 14/15 summaries met the word limit. It invented ID 123456 in NEW-1-12 and missed the financial-impact severity override in NEW-2-07. Schema-constrained decoding did not correct either content error.

## Submission
I submit this completed notebook with visible metrics, full call-log DataFrame, error table, tiny-model comparison and five analysis answers. The accompanying results directory preserves original responses and timings for reproduction. All measurements are from the declared local CPU runtime and CPU track.